In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_2_CVAE import CVAE
from e_1_run_cvae import train_chunk
#from e_1_run_cvae_gpu import train_chunk
#from e_1_run_cvae_time_check import train_chunk
#from e_2_CVAE_norm import CVAE as CVAE_norm
#from e_1_run_cvae_norm import train_chunk_norm

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs_clip' # bs, bs_clip, hes, hes_clip
# barr_type = 'van' # van or barr
# opt_type = 'call' # call or put
# chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
# eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

# if not(opt_type == 'call' or  opt_type == 'put'):
#     raise ValueError("option_type must be 'call' or 'put'")

# if not(barr_type == 'van' or  barr_type == 'barr'):
#     raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs' or model_type == 'bs_clip' or model_type == 'hes_clip'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

# if model_type == 'hes':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.115733
#         else: # put
#             bench_price = 0.005170
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.124491
#         else: # put
#             bench_price = 0.080488

# elif model_type == 'bs':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.123493
#         else: # put
#             bench_price = 0.009535
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.129944
#         else: # put
#             bench_price = 0.085942

/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# training

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 94
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk485.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579.pt"

In [4]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk485.pt | 완료 chunks=485
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=485->579 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=100 | bn_chunks=None | warmup_chunks=None
Chunk step   486 | epoch    6 chunk   1/97 | file_idx  27 | BN off    | beta_eff: 1.0000 | Recon: -5.4227 | KL: 4.6824 | Total: -0.7402
Chunk step   487 | epoch    6 chunk   2/97 | file_idx  82 | BN off    | beta_eff: 1.0000 | Recon: -5.4260 | KL: 4.6760 | Total: -0.7500
Chunk step   488 | epoch    6 chunk   3/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -5.4240 | KL: 4.6892 | Total: -0.7348
Chunk step   489 | epoch    6 chunk   4/97 | file_idx  96 | BN off    | beta_eff: 1.0000 | Recon: -5.4250 | KL: 4.6859 | Total: -0.7391
Chunk step   490 | epoch    6 chunk   5/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.4267 | KL: 4.6874 | Total: -

In [5]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk579.pt | 완료 chunks=579
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=579->582 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None
Chunk step   580 | epoch    6 chunk  95/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.5100 | KL: 4.7661 | Total: -0.7439
Validation @ chunk   580 | Recon: -5.5089 | KL: 4.7701 | Total: -0.7388 | KL_dim: [1.640898, 3.129206]
Chunk step   581 | epoch    6 chunk  96/97 | file_idx  98 | BN off    | beta_eff: 1.0000 | Recon: -5.5103 | KL: 4.7653 | Total: -0.7450
Validation @ chunk   581 | Recon: -5.5081 | KL: 4.7703 | Total: -0.7378 | KL_dim: [1.639696, 3.1306]
Chunk step   582 | epoch    6 chunk  97/97 | file_idx  18 | BN off    | beta_eff: 1.0000 | Recon: -5.5114 | KL: 4.7722 | Total: -0.7392
Validation @ chunk   582 | Recon: -5.5142 | KL: 4.7736 | Total: 

In [18]:
num_chunks  = 92
val_every_chunks = 100
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk873.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk965.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_128_4096_None_0.0001_1_[15, 24, 78]_chunk873.pt | 완료 chunks=873
learning rate : 0.0001
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=873->965 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=100 | bn_chunks=None | warmup_chunks=None
Chunk step   874 | epoch   10 chunk   1/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -4.9769 | KL: 3.7002 | Total: -1.2767
Chunk step   875 | epoch   10 chunk   2/97 | file_idx  44 | BN off    | beta_eff: 1.0000 | Recon: -4.9808 | KL: 3.7036 | Total: -1.2773
Chunk step   876 | epoch   10 chunk   3/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -4.9794 | KL: 3.7050 | Total: -1.2744
Chunk step   877 | epoch   10 chunk   4/97 | file_idx  61 | BN off    | beta_eff: 1.0000 | Recon: -4.9813 | KL: 3.6994 | Total: -1.2820
Chunk step   878 | epoch   10 chunk   5/97 | file_idx  11 | BN off    | beta_eff: 1.0000 | Recon: -4.9756 | KL: 3.7043 | Total: -1.2713

In [19]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk965.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_128_4096_None_0.0001_1_[15, 24, 78]_chunk965.pt | 완료 chunks=965
learning rate : 0.0001
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=965->970 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None
Chunk step   966 | epoch   10 chunk  93/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -4.9845 | KL: 3.7137 | Total: -1.2708
Validation @ chunk   966 | Recon: -4.9737 | KL: 3.7112 | Total: -1.2625 | KL_dim: [1.226394, 2.48479]
Chunk step   967 | epoch   10 chunk  94/97 | file_idx  23 | BN off    | beta_eff: 1.0000 | Recon: -4.9819 | KL: 3.7126 | Total: -1.2693
Validation @ chunk   967 | Recon: -4.9714 | KL: 3.7003 | Total: -1.2711 | KL_dim: [1.22094, 2.479351]
Chunk step   968 | epoch   10 chunk  95/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -4.9855 | KL: 3.7062 | Total: -1.2793
Validation @ chunk   968 | Recon: -4.9734 | KL: 3.7016 | Total: -1.271

In [20]:
num_chunks  = 92
val_every_chunks = 100
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1062.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_128_4096_None_0.0001_1_[15, 24, 78]_chunk970.pt | 완료 chunks=970
learning rate : 0.0001
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=970->1062 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=100 | bn_chunks=None | warmup_chunks=None
Chunk step   971 | epoch   11 chunk   1/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -4.9801 | KL: 3.7126 | Total: -1.2676
Chunk step   972 | epoch   11 chunk   2/97 | file_idx  91 | BN off    | beta_eff: 1.0000 | Recon: -4.9788 | KL: 3.7079 | Total: -1.2710
Chunk step   973 | epoch   11 chunk   3/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -4.9829 | KL: 3.7097 | Total: -1.2732
Chunk step   974 | epoch   11 chunk   4/97 | file_idx  49 | BN off    | beta_eff: 1.0000 | Recon: -4.9804 | KL: 3.7063 | Total: -1.2741
Chunk step   975 | epoch   11 chunk   5/97 | file_idx  54 | BN off    | beta_eff: 1.0000 | Recon: -4.9872 | KL: 3.7123 | Total: -1.274

In [3]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 5
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1062.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_128_4096_None_0.0001_1_[15, 24, 78]_chunk1062.pt | 완료 chunks=1062
learning rate : 0.0001
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=1062->1067 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=5 | bn_chunks=None | warmup_chunks=None
Chunk step  1063 | epoch   11 chunk  93/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -4.9918 | KL: 3.7147 | Total: -1.2771
Chunk step  1064 | epoch   11 chunk  94/97 | file_idx  86 | BN off    | beta_eff: 1.0000 | Recon: -4.9879 | KL: 3.7080 | Total: -1.2799
Chunk step  1065 | epoch   11 chunk  95/97 | file_idx  10 | BN off    | beta_eff: 1.0000 | Recon: -4.9888 | KL: 3.7197 | Total: -1.2691
Validation @ chunk  1065 | Recon: -4.9780 | KL: 3.7164 | Total: -1.2616 | KL_dim: [1.229632, 2.486777]
Chunk step  1066 | epoch   11 chunk  96/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -4.9906 | KL: 3.7150 | Total: -1.2756
Chunk step  1067 | epoch   11 c

In [9]:
num_chunks  = 92
val_every_chunks = 100
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk383.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk291.pt | 완료 chunks=291
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=291->383 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=100 | bn_chunks=None | warmup_chunks=None
Chunk step   292 | epoch    4 chunk   1/97 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -5.3014 | KL: 4.5709 | Total: -0.7304
Chunk step   293 | epoch    4 chunk   2/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.3023 | KL: 4.5654 | Total: -0.7369
Chunk step   294 | epoch    4 chunk   3/97 | file_idx  12 | BN off    | beta_eff: 1.0000 | Recon: -5.3050 | KL: 4.5660 | Total: -0.7389
Chunk step   295 | epoch    4 chunk   4/97 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -5.3000 | KL: 4.5804 | Total: -0.7197
Chunk step   296 | epoch    4 chunk   5/97 | file_idx  94 | BN off    | beta_eff: 1.0000 | Recon: -5.3035 | KL: 4.5789 | Total: -

In [10]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk383.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs_clip/cvae_bs_clip_2_128_4096_None_1e-05_1_[15, 24, 78]_chunk383.pt | 완료 chunks=383
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=383->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None
Chunk step   384 | epoch    4 chunk  93/97 | file_idx   4 | BN off    | beta_eff: 1.0000 | Recon: -5.3433 | KL: 4.5982 | Total: -0.7451
Validation @ chunk   384 | Recon: -5.3385 | KL: 4.6052 | Total: -0.7333 | KL_dim: [1.617292, 2.987899]
Chunk step   385 | epoch    4 chunk  94/97 | file_idx  69 | BN off    | beta_eff: 1.0000 | Recon: -5.3410 | KL: 4.6000 | Total: -0.7410
Validation @ chunk   385 | Recon: -5.3412 | KL: 4.6066 | Total: -0.7346 | KL_dim: [1.616702, 2.989909]
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.3414 | KL: 4.6042 | Total: -0.7372
Validation @ chunk   386 | Recon: -5.3390 | KL: 4.6052 | Total

# BN = 5

In [2]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = 5 # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-5
l2          = 3
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 92
validation_chunk_idxs = [15,24,78]
val_every_chunks = 10
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_0.0001_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=92 | 진행 chunks=194->286 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=10 | bn_chunks=5 | warmup_chunks=None
Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN frozen | beta_eff: 1.0000 | Recon: -5.3391 | KL: 4.5933 | Total: -0.7458
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN frozen | beta_eff: 1.0000 | Recon: -5.4229 | KL: 4.6885 | Total: -0.7345
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN frozen | beta_eff: 1.0000 | Recon: -5.4530 | KL: 4.7225 | Total: -0.7305
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN frozen | beta_eff: 1.0000 | Recon: -5.4805 | KL: 4.7367 | Total: -0.7437
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN frozen | beta_eff: 1.0000 | Recon: -5.4988 | KL: 4.7528 | Total: -0.7460
Chunk ste

KeyboardInterrupt: 

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_128_4096_5_3-0.0001_4-1e-06_1_[15, 24, 78]_chunk383.pt | 완료 chunks=383
학습 시작 | 이번 실행 chunks=5 | 진행 chunks=383->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=5 | warmup_chunks=None
Chunk step   384 | epoch    4 chunk  93/97 | file_idx   4 | BN frozen | beta_eff: 1.0000 | Recon: -5.2304 | KL: 4.4934 | Total: -0.7370
Validation @ chunk   384 | Recon: -5.2282 | KL: 4.4857 | Total: -0.7425 | KL_dim: [1.436707, 3.048991]
Chunk step   385 | epoch    4 chunk  94/97 | file_idx  69 | BN frozen | beta_eff: 1.0000 | Recon: -5.2268 | KL: 4.5010 | Total: -0.7258
Validation @ chunk   385 | Recon: -5.2273 | KL: 4.4858 | Total: -0.7415 | KL_dim: [1.42515, 3.060651]
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN frozen | beta_eff: 1.0000 | Recon: -5.2385 | KL: 4.4922 | Total: -0.7463
Validation @ chunk   386 | Recon: -5.2095 | KL: 4.4750 | Total: -0.7345 | KL_dim: [1.42636

In [ ]:
# CVAE training settings
lr          = 1e-4 # 1e-3, 1e-4와 1e-5는 비슷
l1          = 2
lr2         = 1e-6
l2          = 3
num_chunks  = 92
val_every_chunks = 10
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 5
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk286.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr2,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

# X,M norm

In [ ]:
# CVAE training settings
dim_z       = 8 # 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 0.001 # 0.001,0.0003, 0.0005
beta        = 0.9
warmup_chunks = None # None or num
num_chunks  = 70
validation_chunk_idxs = [15,24,78]
val_every_chunks = 3
resume_path = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk30.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae_xm_norm/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk100.pt"

In [6]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_norm(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    x_mean=bs_stats["x_mean"],
    x_std=bs_stats["x_std"],
    m_mean=bs_stats["m_mean"],
    m_std=bs_stats["m_std"],
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae_xm_norm/bs/cvae_bs_8_128_4096_None_0.001_0.9_None_chunk30.pt | 완료 chunks=30
학습 시작 | 이번 실행 chunks=70 | 진행 chunks=30->100 | files/epoch=100 | bn_chunks=None | warmup_chunks=None
Chunk step    31 | epoch    1 chunk  31/100 | file_idx  30 | BN off    | beta_eff: 0.9000 | Recon: -4.9477 | KL: 5.3827 | Total: -0.1033
Chunk step    32 | epoch    1 chunk  32/100 | file_idx  29 | BN off    | beta_eff: 0.9000 | Recon: -4.8912 | KL: 6.1971 | Total: 0.6862
Chunk step    33 | epoch    1 chunk  33/100 | file_idx  79 | BN off    | beta_eff: 0.9000 | Recon: -4.9538 | KL: 6.2571 | Total: 0.6776
Chunk step    34 | epoch    1 chunk  34/100 | file_idx  44 | BN off    | beta_eff: 0.9000 | Recon: -5.0059 | KL: 5.4327 | Total: -0.1164
Chunk step    35 | epoch    1 chunk  35/100 | file_idx  71 | BN off    | beta_eff: 0.9000 | Recon: -4.9941 | KL: 5.4282 | Total: -0.1088
Chunk step    36 | epoch    1 chunk  36/100 | file_idx  66 | BN off    | beta_eff: 0.9000 | Rec